In [ ]:
!pip install rawpy
from google.colab import drive
import rawpy
from tqdm.notebook import tqdm
import torch
import os
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
def get_image_paths() -> list[str]:
  drive.mount('/content/drive')

  # Define all folders in one list (add more here if needed)
  folders = [
      '/content/drive/MyDrive/MacroInvertebrates Project/Images/SESSION 1',
      '/content/drive/MyDrive/MacroInvertebrates Project/Images/SESSION 2',
      '/content/drive/MyDrive/MacroInvertebrates Project/Images/February 23 & 24 Photos'
  ]

  images = []

  for folder in folders:
      if not os.path.exists(folder):
          raise Exception(f"Folder not found: {folder}")

      folder_count = 0

      # Walk recursively
      for root, _, files in os.walk(folder):
          for file in files:
            if file.lower().endswith(".cr2"):
              full_path = os.path.join(root, file)
              images.append(full_path)
              folder_count += 1

      print(f"{folder} → {folder_count} files found")

  print(f"\nTotal images collected: {len(images)}")

  return images

In [ ]:
def get_all_images():
  # cadis fly = 0 stone fly = 1 mayfly = 2 other =3

  # (num of pictures, species, num of organisms in picture)
  segments = [
      (14, 1, 6),
      (31, 1, 1),
      (11, 0, 10),
      (19, 0, 1),
      (24, 0, 1),
      (12, 2, 2),
      (42, 2, 1),
      (52, 3, 1),
      (40, 1, 10),
      (40, 0, 6),
      (40, 2, 12),
  ]

  species = []
  count = []

  for length, sp, ct in segments:
      species.extend([sp] * length)
      count.extend([ct] * length)

  data = np.column_stack((get_image_paths(), species, count))

  print(data.shape)
  print(data[:5])
  print(data[-5:])

  return data

In [ ]:
images = get_all_images()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/MacroInvertebrates Project/Images/SESSION 1 → 75 files found
/content/drive/MyDrive/MacroInvertebrates Project/Images/SESSION 2 → 130 files found
/content/drive/MyDrive/MacroInvertebrates Project/Images/February 23 & 24 Photos → 120 files found

Total images collected: 325
(325, 3)
[['/content/drive/MyDrive/MacroInvertebrates Project/Images/SESSION 1/5B7A9558.CR2'
  '1' '6']
 ['/content/drive/MyDrive/MacroInvertebrates Project/Images/SESSION 1/5B7A9559.CR2'
  '1' '6']
 ['/content/drive/MyDrive/MacroInvertebrates Project/Images/SESSION 1/5B7A9560.CR2'
  '1' '6']
 ['/content/drive/MyDrive/MacroInvertebrates Project/Images/SESSION 1/5B7A9561.CR2'
  '1' '6']
 ['/content/drive/MyDrive/MacroInvertebrates Project/Images/SESSION 1/5B7A9562.CR2'
  '1' '6']]
[['/content/drive/MyDrive/MacroInvertebrates Project/Images/February 23 & 24 Photos/DCIM/

In [ ]:
train, test = train_test_split(images, test_size=0.2, random_state=1)

In [ ]:
np.save("/content/drive/MyDrive/MacroInvertebrates Project/Save States/train_images.npy", train)
np.save("/content/drive/MyDrive/MacroInvertebrates Project/Save States/test_images.npy", test)

NameError: name 'np' is not defined

In [ ]:
import numpy as np
from google.colab import drive

drive.mount('/content/drive')
train_images = np.load("/content/drive/MyDrive/MacroInvertebrates Project/Save States/train_images.npy")

Mounted at /content/drive


In [ ]:
import os
def raw_to_rgb(raw_img: os.path) -> np.float32:
  # Open the RAW image file using rawpy
  with rawpy.imread(raw_img) as raw:
      # Post-process the RAW image to an RGB format
      rgb = raw.postprocess(
          use_camera_wb=True,
          no_auto_bright=True,
          output_color=rawpy.ColorSpace.sRGB
        )

  return rgb

In [ ]:
!pip install rawpy

import rawpy

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
import numpy as np

In [ ]:
import cv2
from sklearn.model_selection import train_test_split

save_dir = "/content/drive/MyDrive/Macro_Invertebrate_Classifier/Save States"
os.makedirs(save_dir, exist_ok=True)

# Stratified sample of ~25%
indices = list(range(len(train_images)))
species_labels = [species for _, species, _ in train_images]
sample_indices, _ = train_test_split(indices, test_size=0.75, stratify=species_labels, random_state=42)

# Load, resize, save to npz
images = []
for i, idx in enumerate(tqdm(sorted(sample_indices))):
    path, species, count = train_images[idx]
    image = raw_to_rgb(path)
    image = cv2.resize(image, (2016, 1344))  # half of largest dimension
    images.append(image)

np.savez(f"{save_dir}/train_images_sample.npz", Image=np.stack(images))
del images

  0%|          | 0/65 [00:00<?, ?it/s]

In [ ]:
images = np.load("/content/drive/MyDrive/Macro_Invertebrate_Classifier/Save States/train_images_sample.npz")

In [ ]:
print(train_images[0])

['/content/drive/MyDrive/MacroInvertebrates Project/Images/February 23 & 24 Photos/DCIM/100EOS5D/5B7A9996.CR2'
 '1' '10']


In [ ]:
from huggingface_hub import login
login()  # paste your token when prompted

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [1]:
from datasets import Dataset, ClassLabel
from PIL import Image as PILImage
import numpy as np
import rawpy
from tqdm.notebook import tqdm

def raw_to_pil(path):
    with rawpy.imread(path) as raw:
        rgb = raw.postprocess(
            use_camera_wb=True,
            no_auto_bright=True,
            output_color=rawpy.ColorSpace.sRGB
        )
    img = PILImage.fromarray(rgb)
    img = img.resize((2016, 1344), PILImage.LANCZOS)
    return img

# Build dataset
data = {
    "image": [],
    "species": [],
    "count": []
}

for path, species, count in tqdm(train_images):
    try:
        img = raw_to_pil(path)
        data["image"].append(img)
        data["species"].append(int(species))
        data["count"].append(int(count))
    except Exception as e:
        print(f"Skipping {path}: {e}")

# Create dataset, cast species to ClassLabel, split, and push
dataset = Dataset.from_dict(data)
dataset = dataset.cast_column("species", ClassLabel(num_classes=4, names=["0", "1", "2", "3"]))

split = dataset.train_test_split(test_size=0.2, stratify_by_column="species", seed=42)
print(split)

split.push_to_hub("wilbeimer/ept-bioassessment-dataset", private=True)

ModuleNotFoundError: No module named 'datasets'